In [26]:
import os
import torch
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm

In [27]:
image_dir = "artists/artists/resized"  # Directory containing all images
csv_file = "artists/artists/artists.csv"  # CSV file with artist information

In [28]:
artist_df = pd.read_csv(csv_file)

In [29]:
artist_df.head()

,id,name,years,genre,nationality,bio,wikipedia,paintings
0,0,Amedeo Modigliani,1884 - 1920,Expressionism,Italian,Amedeo Clemente Modigliani (Italian pronunciat...,http://en.wikipedia.org/wiki/Amedeo_Modigliani,193
1,1,Vasiliy Kandinskiy,1866 - 1944,"Expressionism,Abstractionism",Russian,Wassily Wassilyevich Kandinsky (Russian: Васи́...,http://en.wikipedia.org/wiki/Wassily_Kandinsky,88
2,2,Diego Rivera,1886 - 1957,"Social Realism,Muralism",Mexican,Diego María de la Concepción Juan Nepomuceno E...,http://en.wikipedia.org/wiki/Diego_Rivera,70
3,3,Claude Monet,1840 - 1926,Impressionism,French,Oscar-Claude Monet (; French: [klod mɔnɛ]; 14 ...,http://en.wikipedia.org/wiki/Claude_Monet,73
4,4,Rene Magritte,1898 - 1967,"Surrealism,Impressionism",Belgian,René François Ghislain Magritte (French: [ʁəne...,http://en.wikipedia.org/wiki/René_Magritte,194


In [30]:
# dictionary to map artist names to numerical labels
artist_to_label = {name: idx for idx, name in enumerate(artist_df['name'])}

In [31]:
# Custom Dataset Class
class ArtistDataset(Dataset):
    def __init__(self, image_dir, artist_to_label, transform=None):
        """
        Args:
            image_dir (str): Directory with all images.
            artist_to_label (dict): Dictionary mapping artist names to numerical labels.
            transform (callable, optional): Optional transform to be applied on a sample.
        """
        self.image_dir = image_dir
        self.image_files = os.listdir(image_dir)  # List all image files in the directory
        self.artist_to_label = artist_to_label
        self.transform = transform

    def __len__(self):
        return len(self.image_files)

    def __getitem__(self, idx):
        # Get the filename at the given index
        image_file = self.image_files[idx]

        # Extract the artist name from the filename (e.g., "Albrecht_Duerer_1.jpg")
        artist_name = image_file.split('_')[0] + '_' + image_file.split('_')[1]

        # Map the artist name to a numerical label
        label = self.artist_to_label[artist_name]

        # Load the image
        image_path = os.path.join(self.image_dir, image_file)
        image = Image.open(image_path).convert("RGB")  # Convert to RGB if needed

        # Apply transformations
        if self.transform:
            image = self.transform(image)

        return image, label

In [32]:
# Image Transformations
transform = transforms.Compose([
    transforms.Resize((224, 224)),  # Resize to match the input size of pre-trained models
    transforms.ToTensor(),  # Convert to PyTorch tensor
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])  # Normalize
])

In [33]:
# Create custom dataset
dataset = ArtistDataset(image_dir, artist_to_label, transform=transform)

In [ ]:
# Split the dataset into training and validation sets
train_size = int(0.7 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    dataset, [train_size, val_size, test_size]
)

In [40]:
# DataLoaders
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4)

In [41]:
# Pre-trained Model (Transfer Learning)
device = torch.device("cuda:0")

model = models.resnet18(pretrained=True)
num_features = model.fc.in_features
model.fc = nn.Linear(num_features, len(artist_to_label))  # Adjust output to number of artists
model = model.to(device)

c:\Users\lukas\Documents\Uni\AAML\venv\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\lukas\Documents\Uni\AAML\venv\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet18_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet18_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)
Downloading: "https://download.pytorch.org/models/resnet18-f37072fd.pth" to C:\Users\lukas/.cache\torch\hub\checkpoints\resnet18-f37072fd.pth
100.0%


In [42]:
# Loss Function and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

In [43]:
def train_model(model, criterion, optimizer, train_loader, val_loader, num_epochs=25):
    best_model_wts = model.state_dict()
    best_acc = 0.0

    for epoch in range(num_epochs):
        print(f'Epoch {epoch}/{num_epochs - 1}')
        print('-' * 10)

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
                loader = train_loader
            else:
                model.eval()
                loader = val_loader

            running_loss = 0.0
            running_corrects = 0

            # Use tqdm to add a progress bar for the current phase
            with tqdm(total=len(loader), desc=f'{phase.capitalize()} Phase', unit='batch') as pbar:
                for inputs, labels in loader:
                    inputs, labels = inputs.to(device), labels.to(device)
                    optimizer.zero_grad()

                    with torch.set_grad_enabled(phase == 'train'):
                        outputs = model(inputs)
                        _, preds = torch.max(outputs, 1)
                        loss = criterion(outputs, labels)

                        if phase == 'train':
                            loss.backward()
                            optimizer.step()

                    running_loss += loss.item() * inputs.size(0)
                    running_corrects += torch.sum(preds == labels.data)

                    # Update the progress bar
                    pbar.update(1)

            epoch_loss = running_loss / len(loader.dataset)
            epoch_acc = running_corrects.double() / len(loader.dataset)

            print(f'{phase} Loss: {epoch_loss:.4f} Acc: {epoch_acc:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = model.state_dict()

    print(f'Best val Acc: {best_acc:.4f}')
    model.load_state_dict(best_model_wts)
    return model

In [ ]:
model = train_model(model, criterion, optimizer, train_loader, val_loader, num_epochs=25)

Epoch 0/24
----------


In [ ]:
# Testing the Model
def test_model(model, test_loader):
    model.eval()  # Set model to evaluation mode
    running_corrects = 0

    with torch.no_grad():  # No gradient computation for testing
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            _, preds = torch.max(outputs, 1)
            running_corrects += torch.sum(preds == labels.data)

    test_acc = running_corrects.double() / len(test_loader.dataset)
    print(f'Test Accuracy: {test_acc:.4f}')

In [ ]:
test_model(model, test_loader)